# Notebook 01: Data Preprocessing & Feature Engineering
**Capstone Project: FITCORE Gym Management & Recommendation System**

This notebook demonstrates:
1. Synthetic training data generation with ACSM-based ground truth labeling
2. BMI feature engineering correlated with fitness goals
3. Feature distributions and class balance analysis
4. Categorical encoding via OneHotEncoder and numerical scaling via StandardScaler

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

print('Libraries loaded successfully.')

## 1. Generate Synthetic Training Data

We generate **1,500 synthetic user records** simulating gym member profiles.

**Why synthetic?** No publicly available dataset maps user biometrics → workout split assignments. Ground truth labels follow **ACSM (American College of Sports Medicine)** exercise physiology guidelines.

**8% label noise** is injected to simulate real-world ambiguity where multiple workout splits could be valid for a single user profile.

In [ ]:
def generate_synthetic_training_data(n_samples=1500, noise_rate=0.08):
    np.random.seed(42)
    goals = ['Weight Loss', 'Muscle Gain', 'Improve Endurance', 'General Fitness']
    genders = ['Male', 'Female', 'Other']
    activity_levels = ['Sedentary', 'Lightly Active', 'Moderately Active', 'Very Active']
    experience_levels = ['Beginner', 'Intermediate', 'Advanced']
    all_splits = [
        'Full-Body Conditioning', 'Full-Body Foundation', 'Push-Pull-Legs',
        'Push-Pull-Legs Advanced', 'Upper-Lower Hypertrophy',
        'Upper-Lower Balanced', 'Cardio-HIIT-Endurance'
    ]

    data = []
    for _ in range(n_samples):
        age = np.random.randint(16, 68)
        gender = np.random.choice(genders, p=[0.48, 0.48, 0.04])
        goal = np.random.choice(goals)
        act = np.random.choice(activity_levels)
        exp = np.random.choice(experience_levels)

        # BMI correlated with fitness goal
        if goal == 'Weight Loss': bmi = np.random.normal(29.5, 4.0)
        elif goal == 'Muscle Gain': bmi = np.random.normal(23.5, 2.5)
        elif goal == 'Improve Endurance': bmi = np.random.normal(22.0, 2.0)
        else: bmi = np.random.normal(24.5, 3.0)
        bmi = float(np.clip(bmi, 16.5, 45.0))

        # ACSM-based ground truth labeling
        if exp == 'Beginner':
            split = 'Full-Body Conditioning' if (goal == 'Weight Loss' or act == 'Sedentary') else 'Full-Body Foundation'
        elif exp == 'Intermediate':
            if goal == 'Muscle Gain': split = 'Push-Pull-Legs'
            elif goal == 'Weight Loss': split = 'Upper-Lower Hypertrophy'
            elif goal == 'Improve Endurance': split = 'Cardio-HIIT-Endurance'
            else: split = 'Upper-Lower Balanced'
        else:
            if goal == 'Muscle Gain': split = 'Push-Pull-Legs Advanced'
            elif goal == 'Improve Endurance': split = 'Cardio-HIIT-Endurance'
            else: split = 'Push-Pull-Legs'

        # Inject noise
        if np.random.random() < noise_rate:
            split = np.random.choice(all_splits)

        data.append({'age': age, 'gender': gender, 'bmi': round(bmi, 1),
                     'fitness_goal': goal, 'activity_level': act,
                     'experience_level': exp, 'workout_split': split})

    return pd.DataFrame(data)

df = generate_synthetic_training_data(1500, noise_rate=0.08)
print(f'Dataset shape: {df.shape}')
df.head(10)

## 2. Dataset Summary Statistics

In [ ]:
print('=== Numerical Feature Statistics ===')
print(df[['age', 'bmi']].describe())
print()
print('=== Target Class Distribution ===')
print(df['workout_split'].value_counts())
print()
print('=== Gender Distribution ===')
print(df['gender'].value_counts())
print()
print('=== Fitness Goal Distribution ===')
print(df['fitness_goal'].value_counts())
print()
print('=== Experience Level Distribution ===')
print(df['experience_level'].value_counts())

## 3. BMI Distribution by Fitness Goal

Demonstrates that BMI is **correlated with fitness goals** — Weight Loss users have higher BMI (mean ~29.5), while Endurance users are lean (mean ~22.0).

In [ ]:
print('=== BMI Statistics Grouped by Fitness Goal ===')
bmi_by_goal = df.groupby('fitness_goal')['bmi'].agg(['mean', 'std', 'min', 'max', 'count'])
print(bmi_by_goal.round(2))

## 4. Feature Engineering — ColumnTransformer Pipeline

The preprocessing pipeline:
- **Numerical features** (`age`, `bmi`) → `StandardScaler` (z-score normalization)
- **Categorical features** (`gender`, `fitness_goal`, `activity_level`, `experience_level`) → `OneHotEncoder` (binary dummy variables)

In [ ]:
X = df[['age', 'gender', 'bmi', 'fitness_goal', 'activity_level', 'experience_level']]
y = df['workout_split']

num_features = ['age', 'bmi']
cat_features = ['gender', 'fitness_goal', 'activity_level', 'experience_level']

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
])

X_transformed = preprocessor.fit_transform(X)
ohe = preprocessor.named_transformers_['cat']
cat_names = list(ohe.get_feature_names_out(cat_features))
all_feature_names = num_features + cat_names

print(f'Original features: {X.shape[1]} columns')
print(f'After preprocessing: {X_transformed.shape[1]} features')
print(f'\nFeature names after encoding:')
for i, name in enumerate(all_feature_names):
    print(f'  [{i:2d}] {name}')

In [ ]:
# Show the first 5 rows after transformation
print('=== First 5 rows of transformed feature matrix ===')
transformed_df = pd.DataFrame(X_transformed.toarray() if hasattr(X_transformed, 'toarray') else X_transformed, columns=all_feature_names)
print(transformed_df.head().to_string())

## Summary

| Item | Value |
|---|---|
| Total Samples | 1,500 |
| Input Features (raw) | 6 (age, gender, bmi, fitness_goal, activity_level, experience_level) |
| Input Features (encoded) | 16 (2 numerical + 14 one-hot encoded) |
| Target Classes | 7 workout split categories |
| Noise Rate | 8% label perturbation |
| BMI Correlation | Goal-dependent Gaussian (Weight Loss: μ=29.5, Endurance: μ=22.0) |